In [1]:
import datetime
import os

import matplotlib.pyplot as plt
import pandas as pd
import pymongo
from dotenv import load_dotenv

In [ ]:
def get_sunspot_data() -> pd.DataFrame:
    """
    Connects to the local Docker MongoDB instance, 
    loads the SILSO sunspot data, and returns it 
    as a Pandas DataFrame.
    """
    
    # Load environment variables from the .env file in the project root
    load_dotenv()

    # Load connection information
    MONGO_USER = os.getenv('MONGO_USER')
    MONGO_PASS = os.getenv('MONGO_PASS')
    MONGO_DB_NAME = os.getenv('MONGO_DB_NAME')
    
    MONGO_HOST = "localhost" 
    MONGO_PORT = 27017
    
    COLLECTION_NAME = "silso_sunspot_daily" 

    client = None
    try:
        # Establish connection
        connection_string = f"mongodb://{MONGO_USER}:{MONGO_PASS}@{MONGO_HOST}:{MONGO_PORT}/"
        client = pymongo.MongoClient(connection_string, serverSelectionTimeoutMS=5000)
        
        # Test the connection
        client.server_info() 
        print(f"Successfully connected to MongoDB (Host: {MONGO_HOST}).")
        
        db = client[MONGO_DB_NAME]
        collection = db[COLLECTION_NAME]

        # Query data
        print(f"Loading data from collection: '{COLLECTION_NAME}'...")
        cursor = collection.find({}).sort("timestamp", 1)
        data = list(cursor)
        
        if not data:
            print("Warning: No data found in the collection.")
            return pd.DataFrame()

        # Convert data to Pandas DataFrame
        # Your ETL stores data in a nested structure (e.g., 'data', 'metadata')
        # pd.json_normalize is the best way to flatten this.
        df = pd.json_normalize(data, sep='_')

        # Clean up the DataFrame
        # Convert 'timestamp' to a real datetime object
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        # Set 'timestamp' as the index (common for time series)
        df = df.set_index('timestamp')
        
        print(f"Successfully loaded {len(df)} documents into a DataFrame.")
        return df

        
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

        
        return pd.DataFrame()

    finally:
        # Close the connection
        if client:
            client.close()
            print("MongoDB connection closed.")

In [ ]:
from space_weather_analysis.database.connection import DatabaseConnection

In [ ]:

load_dotenv()

# Load connection information
MONGO_USER = os.getenv('MONGO_USER')
MONGO_PASS = os.getenv('MONGO_PASS')
MONGO_DB_NAME = os.getenv('MONGO_DB_NAME')

MONGO_HOST = "localhost" 
MONGO_PORT = 27017

COLLECTION_NAME = "silso_sunspot_daily" 


connection_string = f"mongodb://{MONGO_USER}:{MONGO_PASS}@{MONGO_HOST}:{MONGO_PORT}/"
client = pymongo.MongoClient(connection_string, serverSelectionTimeoutMS=5000)

In [3]:
db = DatabaseConnection.get_database()
# collection = db[COLLECTION_NAME]

# cursor = collection.find({}).sort("timestamp", 1)

Could not connect to MongoDB: mongo:27017: [Errno -2] Name or service not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 5.0s, Topology Description: <TopologyDescription id: 6964c4c9e6063ab0b71aacbc, topology_type: Unknown, servers: [<ServerDescription ('mongo', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('mongo:27017: [Errno -2] Name or service not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>
Traceback (most recent call last):
  File "/home/patrick/Studium/HKA/WS25_26/Informatik2/Hausarbeit/space_weather_analysis/src/common/database/connection.py", line 59, in get_database
    cls._client.admin.command("ping")
  File "/home/patrick/Studium/HKA/WS25_26/Informatik2/Hausarbeit/space_weather_analysis/.venv/lib/python3.12/site-packages/pymongo/_csot.py", line 125, in csot_wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/patrick/Stud

In [ ]:
list(cursor)

In [ ]:
# Run this line to get the DataFrame
df_sunspots = get_sunspot_data()

# Display the first few rows
if not df_sunspots.empty:
    print("\nDataFrame Preview:")
    print(df_sunspots.head())

In [ ]:
df_sunspots

In [ ]:
df_sunspots = df_sunspots.dropna()
df_sunspots

In [ ]:
ax = df_sunspots.plot(y="data_sunspot_number", figsize=(20, 10))
for i in range(20):
    timepoint = 1810.6 + 11.1 * i
    year = int(timepoint)
    month = int((timepoint - year) * 12) + 1
    dt = datetime.datetime(year, month, 1)
    ax.axvline(dt, color="red", linestyle="dashed")
plt.show()